# BART + NLI Pipeline for Indonesian Abstractive Summarization

This notebook is designed to run on **Kaggle** (no separate script).

Workflow:
1. Check GPU
2. Install dependencies
3. Convert the Liputan-6 dataset to JSONL
4. Preprocessing
5. Train BART
6. Generate summaries + NLI reranking
7. Evaluate ROUGE and factual consistency

## 1. Check GPU

In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

GPU available: True
GPU name: Tesla T4
VRAM: 15.6 GB


## 2. Install Dependency

In [2]:
!pip install -q transformers datasets evaluate rouge-score accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 78.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26

## 3. Setup Directory

In [3]:
from pathlib import Path
import os

WORK_DIR = Path("./results/thesis_pipeline")
DATA_DIR = WORK_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = WORK_DIR / "outputs" / "bart-baseline"

for d in [DATA_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Working dir:", WORK_DIR)

Working dir: /kaggle/working/thesis_pipeline


## 4. Experiment Configuration

In [ ]:
MODEL_NAME        = "facebook/bart-base"
NLI_MODEL_NAME    = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
SUBSET            = "canonical"   # "canonical" or "xtreme"
TEXT_COLUMN       = "article"     # text column in the Liputan-6 dataset
SUMMARY_COLUMN    = "summary"

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Training hyperparameters
MAX_SOURCE_LENGTH = 256  # reduced from 768 for speed
MAX_TARGET_LENGTH = 128
TRAIN_BATCH_SIZE  = 12
EVAL_BATCH_SIZE   = 12
LEARNING_RATE     = 2e-5
NUM_EPOCHS        = 3
GRAD_ACCUM_STEPS  = 2
WEIGHT_DECAY      = 0.01
WARMUP_RATIO      = 0.1
SEED              = 42

# Generation
NUM_CANDIDATES    = 4
ALPHA             = 0.7

BASE_DATASET = Path("./data/liputan6_raw")

print("Config OK")

## 5. Convert Liputan-6 Dataset to JSONL

Merges individual JSON files into `train.jsonl`, `valid.jsonl`, `test.jsonl`.

In [ ]:
import json

SPLIT_MAP = {"train": "train", "dev": "valid", "test": "test"}

def tokens_to_text(sentences):
    return " ".join(" ".join(sent) for sent in sentences)

for src_split, dst_split in SPLIT_MAP.items():
    split_dir = BASE_DATASET / SUBSET / src_split
    output_file = DATA_DIR / f"{dst_split}.jsonl"

    if output_file.exists():
        print(f"{dst_split}.jsonl already exists, skipping.")
        continue

    if not split_dir.exists():
        print(f"Folder not found: {split_dir}")
        continue

    json_files = sorted(split_dir.glob("*.json"))
    print(f"{src_split}: {len(json_files)} files → {output_file.name}")

    with output_file.open("w", encoding="utf-8") as out:
        for jf in json_files:
            with jf.open("r", encoding="utf-8") as f:
                data = json.load(f)
            row = {
                "id": data["id"],
                TEXT_COLUMN: tokens_to_text(data["clean_article"]),
                SUMMARY_COLUMN: tokens_to_text(data["clean_summary"]),
            }
            out.write(json.dumps(row, ensure_ascii=False) + "\n")

print("\nConversion complete.")

## 6. Preprocessing (Filtering & Normalization)

In [ ]:
import re
import unicodedata
from dataclasses import dataclass, field

WHITESPACE_RE = re.compile(r"\s+")
_CONTROL_CHARS = "".join(chr(c) for c in list(range(0x00, 0x09)) + [0x0B, 0x0C] + list(range(0x0E, 0x20)) + [0x7F])
CONTROL_RE = re.compile("[" + re.escape(_CONTROL_CHARS) + "]")

@dataclass
class PreprocessStats:
    total_rows: int = 0
    kept_rows: int = 0
    dropped_empty: int = 0
    dropped_too_short: int = 0
    dropped_duplicates: int = 0

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = CONTROL_RE.sub(" ", text)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = WHITESPACE_RE.sub(" ", text)
    return text.strip()

def is_valid_pair(document, summary, min_document_chars=100, min_summary_chars=20):
    return bool(document) and bool(summary) and len(document) >= min_document_chars and len(summary) >= min_summary_chars

def preprocess_jsonl(input_file, output_file):
    stats = PreprocessStats()
    seen_pairs = set()
    cleaned_rows = []

    with Path(input_file).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            stats.total_rows += 1
            document = normalize_text(str(row.get(TEXT_COLUMN, "")))
            summary = normalize_text(str(row.get(SUMMARY_COLUMN, "")))

            if not document or not summary:
                stats.dropped_empty += 1
                continue
            if not is_valid_pair(document, summary):
                stats.dropped_too_short += 1
                continue
            pair_key = (document, summary)
            if pair_key in seen_pairs:
                stats.dropped_duplicates += 1
                continue
            seen_pairs.add(pair_key)
            cleaned_row = dict(row)
            cleaned_row[TEXT_COLUMN] = document
            cleaned_row[SUMMARY_COLUMN] = summary
            cleaned_rows.append(cleaned_row)
            stats.kept_rows += 1

    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    with Path(output_file).open("w", encoding="utf-8") as f:
        for row in cleaned_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"{Path(input_file).name} → kept {stats.kept_rows}/{stats.total_rows} rows")
    return stats

for split in ["train", "valid", "test"]:
    preprocess_jsonl(DATA_DIR / f"{split}.jsonl", PROCESSED_DIR / f"{split}.jsonl")

print("\nPreprocessing complete.")

## 7. Training Baseline BART

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

def load_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return Dataset.from_list(rows)

set_seed(SEED)

datasets = DatasetDict(
    train=load_jsonl(PROCESSED_DIR / "train.jsonl"),
    validation=load_jsonl(PROCESSED_DIR / "valid.jsonl"),
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def preprocess_fn(batch):
    model_inputs = tokenizer(
        batch[TEXT_COLUMN], max_length=MAX_SOURCE_LENGTH, truncation=True
    )
    labels = tokenizer(
        text_target=batch[SUMMARY_COLUMN], max_length=MAX_TARGET_LENGTH, truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = datasets.map(
    preprocess_fn,
    batched=True,
    remove_columns=datasets["train"].column_names,
    desc="Tokenizing dataset",
)

total_steps = (len(tokenized["train"]) // (TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    learning_rate=LEARNING_RATE,
    logging_steps=10,        # more frequent than before (50)
    logging_first_step=True,
    disable_tqdm=False,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    warmup_steps=warmup_steps,
    save_total_limit=2,
    load_best_model_at_end=False,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Training complete. Model saved to:", OUTPUT_DIR)

## 8. Generate Summary for BART Baseline (1 candidate, no NLI)

**Fix (P0 #1, TODO.md):** `generate_candidates` previously defaulted to `max_source_length=768` and this default was never overridden at the call sites below, so both the baseline and NLI-reranked generation in this notebook truncated source articles at 768 tokens — inconsistent with the 256-token truncation used for training (Section 4 above) and with `03_ablation_study_alpha.ipynb` / `04_comparison_methods.ipynb`, which both explicitly use 256. This caused Table IV's numbers to diverge from Table VI/IX for the nominally identical α = 0.7 configuration (flagged independently by Reviewer 2 and by code review). Fixed below by changing the default to 256. After fixing, re-run this notebook end-to-end and update `results/main_results.json` and Table IV/V in the paper with the new numbers.

In [ ]:
import math
from transformers import AutoModelForSequenceClassification

class NLIScorer:
    def __init__(self, model_name, device=None):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device).eval()
        id2label = {int(k): v.lower() for k, v in self.model.config.id2label.items()}
        self.entailment_idx = next(i for i, l in id2label.items() if "entail" in l)
        self.contradiction_idx = next(i for i, l in id2label.items() if "contrad" in l)
        self.neutral_idx = next(i for i, l in id2label.items() if "neutral" in l)

    @torch.inference_mode()
    def score(self, premise, hypothesis):
        enc = self.tokenizer(premise, hypothesis, truncation=True, max_length=512, return_tensors="pt")
        enc = {k: v.to(self.device) for k, v in enc.items()}
        probs = torch.softmax(self.model(**enc).logits[0], dim=-1)
        return {
            "entailment": float(probs[self.entailment_idx]),
            "contradiction": float(probs[self.contradiction_idx]),
            "neutral": float(probs[self.neutral_idx]),
        }


class BartSummarizer:
    def __init__(self, model_path, device=None):
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device).eval()

    @torch.inference_mode()
    def generate_candidates(self, document, max_source_length=256, max_target_length=128,
                             min_target_length=32, num_beams=4, num_candidates=4):
        # max_source_length must match MAX_SOURCE_LENGTH (Section 4) and the truncation
        # used in 03_ablation_study_alpha.ipynb / 04_comparison_methods.ipynb (TODO.md P0 #1).
        inputs = self.tokenizer(document, truncation=True, max_length=max_source_length, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        generated = self.model.generate(
            **inputs,
            max_length=max_target_length,
            min_length=min_target_length,
            num_beams=max(num_beams, num_candidates),
            num_return_sequences=num_candidates,
            length_penalty=1.0,
            early_stopping=True,
            output_scores=True,
            return_dict_in_generate=True,
        )
        texts = self.tokenizer.batch_decode(generated.sequences, skip_special_tokens=True,
                                             clean_up_tokenization_spaces=True)
        return [{"summary": t.strip(), "generation_score": float(s)}
                for t, s in zip(texts, generated.sequences_scores.tolist())]

    def rerank_with_nli(self, document, candidates, nli_scorer, alpha=0.7):
        best = None
        for item in candidates:
            nli = nli_scorer.score(document, item["summary"])
            norm_gen = math.tanh(item["generation_score"] / 10.0)
            combined = alpha * nli["entailment"] + (1 - alpha) * norm_gen
            enriched = {**item, **nli, "combined_score": combined}
            if best is None or enriched["combined_score"] > best["combined_score"]:
                best = enriched
        return best


def generate_summaries(model_path, input_file, output_file, nli_model_name,
                        num_candidates=1, alpha=0.0):
    summarizer = BartSummarizer(model_path)
    nli_scorer = NLIScorer(nli_model_name)
    outputs = []

    with Path(input_file).open("r", encoding="utf-8") as f:
        rows = [json.loads(l) for l in f if l.strip()]

    for i, row in enumerate(rows):
        document = row[TEXT_COLUMN]
        candidates = summarizer.generate_candidates(
            document, num_candidates=num_candidates
        )
        if num_candidates == 1:
            best = {**candidates[0], "entailment": 0.0, "contradiction": 0.0,
                    "neutral": 0.0, "combined_score": 0.0}
        else:
            best = summarizer.rerank_with_nli(document, candidates, nli_scorer, alpha)
        outputs.append({
            "id": row.get("id"),
            "document": document,
            "reference_summary": row.get(SUMMARY_COLUMN),
            "generated_summary": best["summary"],
            "generation_score": best["generation_score"],
            "entailment_score": best.get("entailment", 0.0),
            "contradiction_score": best.get("contradiction", 0.0),
            "neutral_score": best.get("neutral", 0.0),
            "combined_score": best.get("combined_score", 0.0),
        })
        if (i + 1) % 100 == 0:
            print(f"Generated {i+1}/{len(rows)}")

    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    with Path(output_file).open("w", encoding="utf-8") as f:
        for row in outputs:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Saved {len(outputs)} predictions → {output_file}")


# Baseline: 1 candidate, no NLI reranking
generate_summaries(
    model_path=str(OUTPUT_DIR),
    input_file=PROCESSED_DIR / "test.jsonl",
    output_file=WORK_DIR / "outputs" / "predictions_baseline.jsonl",
    nli_model_name=NLI_MODEL_NAME,
    num_candidates=1,
    alpha=0.0,
)

## 9. Evaluate BART Baseline

In [9]:
import evaluate
import statistics

def evaluate_predictions(prediction_file, nli_model_name):
    rouge = evaluate.load("rouge")
    nli_scorer = NLIScorer(nli_model_name)

    with Path(prediction_file).open("r", encoding="utf-8") as f:
        rows = [json.loads(l) for l in f if l.strip()]

    predictions = [r["generated_summary"] for r in rows]
    references  = [r["reference_summary"] for r in rows]
    rouge_scores = rouge.compute(predictions=predictions, references=references, use_stemmer=False)

    entailments, contradictions = [], []
    for i, row in enumerate(rows):
        result = nli_scorer.score(row["document"], row["generated_summary"])
        entailments.append(result["entailment"])
        contradictions.append(result["contradiction"])
        if (i + 1) % 100 == 0:
            print(f"NLI scored {i+1}/{len(rows)}")

    print("\nROUGE evaluation")
    print(f"ROUGE-1: {rouge_scores['rouge1']:.4f}")
    print(f"ROUGE-2: {rouge_scores['rouge2']:.4f}")
    print(f"ROUGE-L: {rouge_scores['rougeL']:.4f}")
    print("\nFactual consistency evaluation")
    print(f"Average entailment:    {statistics.mean(entailments):.4f}")
    print(f"Average contradiction: {statistics.mean(contradictions):.4f}")

evaluate_predictions(
    prediction_file=WORK_DIR / "outputs" / "predictions_baseline.jsonl",
    nli_model_name=NLI_MODEL_NAME,
)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NLI scored 100/10971
NLI scored 200/10971
NLI scored 300/10971
NLI scored 400/10971
NLI scored 500/10971
NLI scored 600/10971
NLI scored 700/10971
NLI scored 800/10971
NLI scored 900/10971
NLI scored 1000/10971
NLI scored 1100/10971
NLI scored 1200/10971
NLI scored 1300/10971
NLI scored 1400/10971
NLI scored 1500/10971
NLI scored 1600/10971
NLI scored 1700/10971
NLI scored 1800/10971
NLI scored 1900/10971
NLI scored 2000/10971
NLI scored 2100/10971
NLI scored 2200/10971
NLI scored 2300/10971
NLI scored 2400/10971
NLI scored 2500/10971
NLI scored 2600/10971
NLI scored 2700/10971
NLI scored 2800/10971
NLI scored 2900/10971
NLI scored 3000/10971
NLI scored 3100/10971
NLI scored 3200/10971
NLI scored 3300/10971
NLI scored 3400/10971
NLI scored 3500/10971
NLI scored 3600/10971
NLI scored 3700/10971
NLI scored 3800/10971
NLI scored 3900/10971
NLI scored 4000/10971
NLI scored 4100/10971
NLI scored 4200/10971
NLI scored 4300/10971
NLI scored 4400/10971
NLI scored 4500/10971
NLI scored 4600/109

## 10. Generate Summary + NLI Reranking

In [10]:
generate_summaries(
    model_path=str(OUTPUT_DIR),
    input_file=PROCESSED_DIR / "test.jsonl",
    output_file=WORK_DIR / "outputs" / "predictions_nli.jsonl",
    nli_model_name=NLI_MODEL_NAME,
    num_candidates=NUM_CANDIDATES,
    alpha=ALPHA,
)

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generated 100/10971
Generated 200/10971
Generated 300/10971
Generated 400/10971
Generated 500/10971
Generated 600/10971
Generated 700/10971
Generated 800/10971
Generated 900/10971
Generated 1000/10971
Generated 1100/10971
Generated 1200/10971
Generated 1300/10971
Generated 1400/10971
Generated 1500/10971
Generated 1600/10971
Generated 1700/10971
Generated 1800/10971
Generated 1900/10971
Generated 2000/10971
Generated 2100/10971
Generated 2200/10971
Generated 2300/10971
Generated 2400/10971
Generated 2500/10971
Generated 2600/10971
Generated 2700/10971
Generated 2800/10971
Generated 2900/10971
Generated 3000/10971
Generated 3100/10971
Generated 3200/10971
Generated 3300/10971
Generated 3400/10971
Generated 3500/10971
Generated 3600/10971
Generated 3700/10971
Generated 3800/10971
Generated 3900/10971
Generated 4000/10971
Generated 4100/10971
Generated 4200/10971
Generated 4300/10971
Generated 4400/10971
Generated 4500/10971
Generated 4600/10971
Generated 4700/10971
Generated 4800/10971
G

## 11. Evaluate BART + NLI

In [11]:
evaluate_predictions(
    prediction_file=WORK_DIR / "outputs" / "predictions_nli.jsonl",
    nli_model_name=NLI_MODEL_NAME,
)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NLI scored 100/10971
NLI scored 200/10971
NLI scored 300/10971
NLI scored 400/10971
NLI scored 500/10971
NLI scored 600/10971
NLI scored 700/10971
NLI scored 800/10971
NLI scored 900/10971
NLI scored 1000/10971
NLI scored 1100/10971
NLI scored 1200/10971
NLI scored 1300/10971
NLI scored 1400/10971
NLI scored 1500/10971
NLI scored 1600/10971
NLI scored 1700/10971
NLI scored 1800/10971
NLI scored 1900/10971
NLI scored 2000/10971
NLI scored 2100/10971
NLI scored 2200/10971
NLI scored 2300/10971
NLI scored 2400/10971
NLI scored 2500/10971
NLI scored 2600/10971
NLI scored 2700/10971
NLI scored 2800/10971
NLI scored 2900/10971
NLI scored 3000/10971
NLI scored 3100/10971
NLI scored 3200/10971
NLI scored 3300/10971
NLI scored 3400/10971
NLI scored 3500/10971
NLI scored 3600/10971
NLI scored 3700/10971
NLI scored 3800/10971
NLI scored 3900/10971
NLI scored 4000/10971
NLI scored 4100/10971
NLI scored 4200/10971
NLI scored 4300/10971
NLI scored 4400/10971
NLI scored 4500/10971
NLI scored 4600/109

## 12. View Example Predictions

In [12]:
prediction_path = WORK_DIR / "outputs" / "predictions_nli.jsonl"
with prediction_path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        row = json.loads(line)
        print("ID:", row.get("id"))
        print("Generated:", row.get("generated_summary"))
        print("Reference:", row.get("reference_summary"))
        print("Entailment:", row.get("entailment_score"))
        print("-" * 80)
        if i >= 2:
            break

ID: 13019
Generated: Polda Riau akan memberangus menipulasi dana reboisasi dan iuran hasil hutan. Selain itu, Kapolri juga melantik Inspektur Jenderal Polisi Firman Gani dan Brigjen Pol. Eddy Darnadi.
Reference: Kapolda Riau baru Brigjen Pol . Johny Yodjana bertekad memberantas pelaku penyelundupan kayu di Riau . Ia berjanji akan menindak tegas pelaku tanpa pandang bulu .
Entailment: 0.9951171875
--------------------------------------------------------------------------------
ID: 13020
Generated: Bank Indonesia dinilai masih akan menghadapi situasi sulit kendati Bank Sentral Amerika Serikat ( The FED ) terus menurunkan tingkat suku bunga yang dimiliki. Penilaian ini dikemukakan pengamat ekonomi Didiek J. Rachbini.
Reference: Kendati Bank Sentral AS menurunkan suku bunganya , namun BI dinilai masih akan menemui masa sulit . Suku bunga Bank Sentral AS akan diturunkan menjadi empat persen .
Entailment: 0.10809326171875
----------------------------------------------------------------------

## 13. Experiment Notes

For paper reporting, at minimum compare:
- BART baseline
- BART + NLI reranking

Also record:
- the summarization model used
- the NLI model used
- data subset (`canonical` or `xtreme`)
- number of summary candidates
- the `alpha` value
- ROUGE and entailment results

**Kaggle notes:**
- Download results from the **Output** tab after the session ends
- Kaggle provides **30 GPU hours/week** — use them efficiently
- Enable **"Save & Run All (Commit)"** so outputs are saved permanently